In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\soaha\Desktop\ML-Projects\House_price_prediction_endToEnd\.venv\Scripts\python.exe
3.0.4
c:\Users\soaha\Desktop\ML-Projects\House_price_prediction_endToEnd\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\soaha\Desktop\ML-Projects\House_price_prediction_endToEnd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("C:\\Users\\soaha\\Desktop\\ML-Projects\\House_price_prediction_endToEnd\\data\\processed\\feature_engineered_train.csv")
eval_df = pd.read_csv("C:\\Users\\soaha\\Desktop\\ML-Projects\\House_price_prediction_endToEnd\\data\\processed\\feature_engineered_eval.csv")


# Define target + features
target = "price"
cols_to_drop = [target, "median_list_price", "median_ppsf", "median_list_ppsf"]
X_train, y_train = train_df.drop(columns=cols_to_drop), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=cols_to_drop), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576853, 34)
Eval shape: (148448, 34)


In [6]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [10]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
mlflow.set_tracking_uri(
    "file:///C:/Users/soaha/Desktop/ML-Projects/House_price_prediction_endToEnd/mlruns"
)

mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

2026/01/27 00:17:17 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.


[I 2026-01-27 00:17:17,976] A new study created in memory with name: no-name-30565b37-efb4-4822-a11a-6fa1c1fc18f2
[I 2026-01-27 00:18:12,255] Trial 0 finished with value: 124495.00325583483 and parameters: {'n_estimators': 479, 'max_depth': 10, 'learning_rate': 0.01036246097341989, 'subsample': 0.7898827592915434, 'colsample_bytree': 0.7860486981084889, 'min_child_weight': 3, 'gamma': 4.8203908352633436, 'reg_alpha': 0.0007352510740466332, 'reg_lambda': 0.002319138431544221}. Best is trial 0 with value: 124495.00325583483.
[I 2026-01-27 00:19:15,997] Trial 1 finished with value: 117452.61648278983 and parameters: {'n_estimators': 972, 'max_depth': 8, 'learning_rate': 0.017505644917590924, 'subsample': 0.6473181531211972, 'colsample_bytree': 0.6324831522166274, 'min_child_weight': 5, 'gamma': 3.6841393926950223, 'reg_alpha': 3.2492298224192045e-05, 'reg_lambda': 6.29012140860148e-05}. Best is trial 1 with value: 117452.61648278983.
[I 2026-01-27 00:19:36,210] Trial 2 finished with value

Best params: {'n_estimators': 872, 'max_depth': 8, 'learning_rate': 0.11146989588520756, 'subsample': 0.8339009985121654, 'colsample_bytree': 0.6583222270210732, 'min_child_weight': 7, 'gamma': 1.1075809096284337, 'reg_alpha': 0.022566460692215802, 'reg_lambda': 7.450537872787284}


In [11]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 58753.95179641745
RMSE: 113043.19763990105
R²: 0.901247318107463


c:\Users\soaha\Desktop\ML-Projects\House_price_prediction_endToEnd\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [00:30:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/01/27 00:30:58 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/01/27 00:30:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
